# 07.12 Test Methods — Deep Dive

Twelve methods answer yes/no questions about a string. All return `bool`, all
take no arguments, and **all return `False` for an empty string** except
`isprintable()`.

| Method | True when every character is… |
|---|---|
| `isalpha()` | a letter |
| `isdigit()` | a digit |
| `isalnum()` | a letter or digit |
| `isspace()` | whitespace |
| `isnumeric()` | numeric (wider than `isdigit`) |
| `isdecimal()` | a decimal digit (narrower than `isdigit`) |
| `isascii()` | in the ASCII range |
| `isprintable()` | printable |
| `isupper()` | uppercase (ignoring non-letters) |
| `islower()` | lowercase (ignoring non-letters) |
| `istitle()` | following title-case rules |
| `isidentifier()` | valid as a Python name |

The three numeric ones — `isdecimal`, `isdigit`, `isnumeric` — are a strict
hierarchy that almost nobody knows.

## Easy — The basic checks

One question per cell.

In [ ]:
# EXAMPLE 1: isalpha() — is it all letters?
print("'hello'.isalpha() ->", "hello".isalpha())
print("'hello1'.isalpha() ->", "hello1".isalpha())

In [ ]:
# EXAMPLE 2: isdigit() — is it all digits?
print("'123'.isdigit() ->", "123".isdigit())
print("'12a'.isdigit() ->", "12a".isdigit())

In [ ]:
# EXAMPLE 3: isalnum() — letters or digits
print("'abc123'.isalnum() ->", "abc123".isalnum())
print("'abc 123'.isalnum() ->", "abc 123".isalnum())

In [ ]:
# EXAMPLE 4: isspace() — is it all whitespace?
print("'   '.isspace() ->", "   ".isspace())
print("'a b'.isspace() ->", "a b".isspace())

In [ ]:
# EXAMPLE 5: isupper() and islower()
print("'HELLO'.isupper() ->", "HELLO".isupper())
print("'hello'.islower() ->", "hello".islower())
print("'Hello'.isupper() ->", "Hello".isupper())

In [ ]:
# EXAMPLE 6: istitle() — first letter of each word capitalised
print("'Hello World'.istitle() ->", "Hello World".istitle())
print("'Hello world'.istitle() ->", "Hello world".istitle())

In [ ]:
# EXAMPLE 7: Every method returns a real bool
result = "abc".isalpha()

print("value:", result)
print("type: ", type(result).__name__)

In [ ]:
# EXAMPLE 8: Empty strings are almost always False
empty = ""

print("isalpha:  ", empty.isalpha())
print("isdigit:  ", empty.isdigit())
print("isalnum:  ", empty.isalnum())
print("isspace:  ", empty.isspace())
print("isupper:  ", empty.isupper())
print("isprintable:", empty.isprintable(), "<- the exception")

In [ ]:
# EXAMPLE 9: Why the empty-string rule matters
# Always check the string is non-empty first.
user_input = ""

if user_input.isdigit():
    print("valid number")
else:
    print("rejected - and note an empty string fails, which is correct here")

## Medium — The methods nobody teaches

Seven methods that almost never appear in tutorials.

In [ ]:
# EXAMPLE 10: isascii() — is every character ASCII?
print("'hello'.isascii() ->", "hello".isascii())
print("'café'.isascii()  ->", "café".isascii())
print("''.isascii()      ->", "".isascii(), "<- empty IS ascii")

In [ ]:
# EXAMPLE 11: Using isascii() for validation
# Useful when a system downstream cannot handle non-ASCII.
names = ["Asha", "José", "北京"]

for name in names:
    status = "ok" if name.isascii() else "needs transliteration"
    print(f"{name:<8} {status}")

In [ ]:
# EXAMPLE 12: isprintable() — can it be displayed?
print("'hello'.isprintable()  ->", "hello".isprintable())
print("'hello\\n'.isprintable() ->", "hello\n".isprintable())
print("'hello world'.isprintable() ->", "hello world".isprintable())
print("")
print("A space IS printable. A newline is not.")

In [ ]:
# EXAMPLE 13: Finding non-printable characters
# Useful for detecting corrupted or malicious input.
import unicodedata

text = "hello\x00world\x07"

if not text.isprintable():
    print("Contains non-printable characters:")
    for index, character in enumerate(text):
        if not character.isprintable():
            name = unicodedata.name(character, "unnamed")
            print(f"   position {index}: {character!r} ({name})")

In [ ]:
# EXAMPLE 14: isidentifier() — is it a valid Python name?
candidates = ["total", "_private", "total2", "2total", "my-var", "class"]

for candidate in candidates:
    print(f"{candidate!r:<12} isidentifier={candidate.isidentifier()}")

In [ ]:
# EXAMPLE 15: isidentifier() does not check keywords
# A keyword IS a valid identifier shape, but cannot be used as a name.
import keyword

candidate = "class"

print("isidentifier:", candidate.isidentifier())
print("is a keyword:", keyword.iskeyword(candidate))
print("")
print("A safe check needs both:")
usable = candidate.isidentifier() and not keyword.iskeyword(candidate)
print("usable as a name:", usable)

In [ ]:
# EXAMPLE 16: A practical use for isidentifier()
# Validating column names from a spreadsheet before using them as attributes.
import keyword


def is_safe_name(text):
    """Check a string can be used as a Python attribute name."""
    return text.isidentifier() and not keyword.iskeyword(text)


columns = ["user_id", "first name", "class", "2024_total", "amount"]

for column in columns:
    print(f"{column!r:<15} {'ok' if is_safe_name(column) else 'needs renaming'}")

In [ ]:
# EXAMPLE 17: The three numeric methods are a hierarchy
# isdecimal is narrowest, isnumeric is widest.
samples = [
    ("123", "plain digits"),
    ("\u00b2", "superscript two"),
    ("\u00bd", "one half"),
    ("\u0f33", "tibetan half zero"),
]

print(f"{'char':<8} {'decimal':<9} {'digit':<7} {'numeric':<9} description")
print("-" * 56)
for text, description in samples:
    print(f"{text:<8} {str(text.isdecimal()):<9} {str(text.isdigit()):<7} "
          f"{str(text.isnumeric()):<9} {description}")

In [ ]:
# EXAMPLE 18: What the hierarchy means
print("isdecimal()  subset of  isdigit()  subset of  isnumeric()")
print("")
print("isdecimal: characters usable in base-10 numbers  (0-9 and equivalents)")
print("isdigit:   also superscripts and similar         (adds \u00b2)")
print("isnumeric: also fractions and numeric characters (adds \u00bd)")
print("")
print("For 'can int() parse this?', isdecimal is the closest match.")

In [ ]:
# EXAMPLE 19: Which one should you use for input validation?
# The honest answer: none of them - use try/except.
candidates = ["42", "\u00b2", "-5", "3.14", "1e3"]

print(f"{'input':<8} {'isdecimal':<11} {'int() works':<13} {'float() works'}")
print("-" * 48)

for text in candidates:
    try:
        int(text)
        int_ok = True
    except ValueError:
        int_ok = False

    try:
        float(text)
        float_ok = True
    except ValueError:
        float_ok = False

    print(f"{text:<8} {str(text.isdecimal()):<11} {str(int_ok):<13} {float_ok}")

print("")
print("Note isdecimal says False for '-5' but int('-5') works fine.")

In [ ]:
# EXAMPLE 20: The correct way to validate a number
def parse_number(text):
    """Convert text to a number, or report why it failed."""
    try:
        return int(text), None
    except ValueError:
        pass

    try:
        return float(text), None
    except ValueError:
        return None, f"{text!r} is not a number"


for candidate in ["42", "3.14", "-5", "1e3", "abc", ""]:
    value, error = parse_number(candidate)
    print(f"{candidate!r:<8} -> {value if error is None else error}")

In [ ]:
# EXAMPLE 21: isupper() and islower() ignore non-letters
# Only cased characters count towards the answer.
samples = ["HELLO", "HELLO123", "HELLO!", "123", "!!!"]

for text in samples:
    print(f"{text!r:<12} isupper={text.isupper()}")

print("")
print("'123' and '!!!' are False - there are no cased characters at all.")

## Hard — Unicode behaviour and real validation

Where these methods do not mean what you assume.

In [ ]:
# EXAMPLE 22: isalpha() accepts every alphabet
samples = ["hello", "नमस्ते", "こんにちは", "Привет", "مرحبا"]

for text in samples:
    print(f"{text:<12} isalpha={text.isalpha()}")

print("")
print("This is correct behaviour, but surprising if you expected A-Z only.")

In [ ]:
# EXAMPLE 23: Restricting to ASCII letters
# Combine two checks when you genuinely mean A-Z.
samples = ["hello", "café", "Привет"]

for text in samples:
    ascii_only = text.isalpha() and text.isascii()
    print(f"{text:<10} isalpha={text.isalpha()}  ascii-only={ascii_only}")

In [ ]:
# EXAMPLE 24: isdigit() accepts non-ASCII digits
# Arabic-Indic digits are digits, and int() parses them.
arabic_digits = "\u0664\u0662"

print("text:", arabic_digits)
print("isdigit:", arabic_digits.isdigit())
print("int() gives:", int(arabic_digits))
print("")
print("Usually fine. A problem only if you assumed ASCII.")

In [ ]:
# EXAMPLE 25: A security-relevant example
# Homograph characters pass isalpha() but are not what they look like.
import unicodedata

latin = "paypal"
mixed = "pay" + "\u0440" + "al"

print("latin:", latin, "isalpha:", latin.isalpha(), "isascii:", latin.isascii())
print("mixed:", mixed, "isalpha:", mixed.isalpha(), "isascii:", mixed.isascii())
print("")
print("The fourth character of the second string:")
print("  ", unicodedata.name(mixed[3]))
print("")
print("isascii() catches this. isalpha() alone does not.")

In [ ]:
# EXAMPLE 26: isspace() and the whitespace characters
# Many characters count as whitespace.
import unicodedata

candidates = [" ", "\t", "\n", "\r", "\u00a0", "\u2003"]

for character in candidates:
    name = unicodedata.name(character, "unnamed")
    print(f"{character!r:<10} isspace={character.isspace():<6} {name}")

In [ ]:
# EXAMPLE 27: The non-breaking space trap
# A non-breaking space looks like a space but is not stripped by default.
normal = "hello world"
nbsp = "hello\u00a0world"

print("look identical:", normal, "|", nbsp)
print("equal?", normal == nbsp)
print("")
print("split() DOES treat it as whitespace:")
print("  normal:", normal.split())
print("  nbsp:  ", nbsp.split())

In [ ]:
# EXAMPLE 28: istitle() and apostrophes
# istitle() uses the same rule as title(), including the odd cases.
candidates = ["O'Brien", "O'brien", "Mother-In-Law", "Mother-in-law"]

for text in candidates:
    print(f"{text!r:<18} istitle={text.istitle()}")

print("")
print("This is consistent with title(), but rarely what a human means.")

In [ ]:
# EXAMPLE 29: Building a real password validator
import string


def check_password(password):
    """Return a list of problems with a password."""
    problems = []

    if len(password) < 8:
        problems.append("too short - needs 8 characters")

    if not any(character.isupper() for character in password):
        problems.append("needs an uppercase letter")

    if not any(character.islower() for character in password):
        problems.append("needs a lowercase letter")

    if not any(character.isdigit() for character in password):
        problems.append("needs a digit")

    if not any(character in string.punctuation for character in password):
        problems.append("needs a symbol")

    if not password.isprintable():
        problems.append("contains non-printable characters")

    return problems


for candidate in ["abc", "Password1!", "password1!", "PASSWORD1!"]:
    problems = check_password(candidate)
    status = "OK" if not problems else "; ".join(problems)
    print(f"{candidate!r:<16} {status}")

In [ ]:
# EXAMPLE 30: Building a username validator
def check_username(name):
    """Validate a username against practical rules."""
    if not name:
        return "cannot be empty"
    if not name.isascii():
        return "ASCII characters only"
    if not name[0].isalpha():
        return "must start with a letter"
    if not all(character.isalnum() or character == "_" for character in name):
        return "letters, digits and underscore only"
    if len(name) > 20:
        return "too long"
    return "OK"


for candidate in ["asha", "asha_99", "9asha", "asha-99", "josé", ""]:
    print(f"{candidate!r:<12} {check_username(candidate)}")

In [ ]:
# EXAMPLE 31: Complete comparison table
samples = ["abc", "ABC", "123", "a1", "  ", "", "Hello World", "café", "_x"]

headers = ["alpha", "digit", "alnum", "space", "upper", "lower",
           "title", "ascii", "ident"]

print(f"{'value':<14} " + " ".join(h[:5].ljust(6) for h in headers))
print("-" * 72)

for text in samples:
    results = [
        text.isalpha(), text.isdigit(), text.isalnum(), text.isspace(),
        text.isupper(), text.islower(), text.istitle(), text.isascii(),
        text.isidentifier(),
    ]
    row = " ".join(str(result)[:5].ljust(6) for result in results)
    print(f"{text!r:<14} {row}")

## Takeaways

1. All twelve return `bool`, take no arguments, and return **`False` for an empty
   string** — except `isprintable()`, which returns `True`.
2. **`isdecimal` ⊂ `isdigit` ⊂ `isnumeric`.** For "can `int()` parse this?",
   `isdecimal` is closest — but still wrong for `-5`.
3. **Do not validate numbers with `isdigit()`.** Use `try: int(text)` — it
   handles signs, whitespace and underscores correctly.
4. `isalpha()` accepts **every alphabet**, not just A–Z. Add `isascii()` when you
   mean ASCII.
5. `isupper()`/`islower()` **ignore non-letters** — `"123".isupper()` is `False`
   because there are no cased characters.
6. **`isidentifier()` does not check keywords** — combine with
   `keyword.iskeyword()`.
7. `isascii()` is a cheap defence against **homograph attacks**.
8. `isspace()` includes the **non-breaking space**, which looks like a space but
   is a different character.

## Try it yourself

1. Run all twelve methods on `""`, `"abc"`, `"123"` and `"  "`.
2. Find a character where `isdigit()` is `True` but `isdecimal()` is `False`.
3. Write a validator accepting only ASCII letters and digits.
4. Explain why `"-5".isdigit()` is `False` but `int("-5")` works.
5. Check whether `"class"` is a usable Python name.